# Drain parser for log analysis

In [6]:
import os, sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))  # project root

# ── Run tag ───────────────────────────────────────────────────────────────────
# Auto-generated as YYYYMMDD_HHMM so multiple daily runs stay distinct.
# Override before running to reload or continue a specific experiment:
#   RUN_TAG = "20260407_1030"   # exact timestamp
#   RUN_TAG = "20260407_v3"     # manual version label
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")

# Convenience: also expose individual parts if needed downstream
RUN_DATE = RUN_TAG.split("_")[0]

print(f"Run tag : {RUN_TAG}")
print(f"Outputs will be prefixed with: {RUN_TAG}_")


Run tag : 20260921_2108
Outputs will be prefixed with: 20260921_2108_


In [7]:
print(f"Paths will be relative to: {Path.cwd()}")


Paths will be relative to: /Users/michalklos/Magisterka 2026/Repository/hybrid-logs-analyzer-research/notebooks


In [8]:
from modules.parser import BGLParser

config_path = '../configs/drain_bgl.ini'
bgl_log_path = "../data/raw/bgl/BGL_full.log"

n_lines = sum(1 for _ in open(bgl_log_path, "rb"))
print(f"Total lines in log: {n_lines:,}")

parser = BGLParser(config_path=config_path)

# First pass: fit the parser to the log file, building the template clusters
print("Fitting parser to log file...")
parser.fit_file(bgl_log_path)

# Second pass: annotate every line with its final cluster_id, template, params, block_id
print("Annotating log file...")
df = parser.annotate_file(bgl_log_path, max_lines=None)
print(df.shape)
df.head(3)

parser.export_templates(f"../data/processed/bgl/{RUN_TAG}/1_bgl_templates.json")
parser.save(f"../models/bgl/{RUN_TAG}/1_drain_parser.bin")


Total lines in log: 4,631,261
Fitting parser to log file...
[INFO] Training Drain3 on: ../data/raw/bgl/BGL_full.log
[INFO] Processed 100000 lines...
[INFO] Processed 200000 lines...
[INFO] Processed 300000 lines...
[INFO] Processed 400000 lines...
[INFO] Processed 500000 lines...
[INFO] Processed 600000 lines...
[INFO] Processed 700000 lines...
[INFO] Processed 800000 lines...
[INFO] Processed 900000 lines...
[INFO] Processed 1000000 lines...
[INFO] Processed 1100000 lines...
[INFO] Processed 1200000 lines...
[INFO] Processed 1300000 lines...
[INFO] Processed 1400000 lines...
[INFO] Processed 1500000 lines...
[INFO] Processed 1600000 lines...
[INFO] Processed 1700000 lines...
[INFO] Processed 1800000 lines...
[INFO] Processed 1900000 lines...
[INFO] Processed 2000000 lines...
[INFO] Processed 2100000 lines...
[INFO] Processed 2200000 lines...
[INFO] Processed 2300000 lines...
[INFO] Processed 2400000 lines...
[INFO] Processed 2500000 lines...
[INFO] Processed 2600000 lines...
[INFO] Pr

## Persistence

In [9]:
import json
from pathlib import Path

PROCESSED_DIR = Path("../data/processed/bgl")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / f"{RUN_TAG}").mkdir(parents=True, exist_ok=True)
PARQUET_PATH  = PROCESSED_DIR / f"{RUN_TAG}" / f"1_parser_bgl_annotated.parquet"

# Parquet cannot natively store Python lists in all engines,
# so encode the `parameters` column as a JSON string.
df_save = df.copy()
df_save["parameters"] = df_save["parameters"].apply(json.dumps)

# fastparquet cannot write pandas Arrow-backed string columns (ArrowDtype).
# The error "Unable to avoid copy while creating an array as requested" is
# caused by those columns. Cast all string/object columns to plain numpy object.
for col in df_save.select_dtypes(include=["object", "string"]).columns:
    df_save[col] = df_save[col].astype(object)

# Use fastparquet to avoid the pandas/pyarrow version-conflict bug
# (ArrowKeyError: arrow.py_extension_type) that occurs when pyarrow
# is loaded after pandas in the same kernel session.
df_save.to_parquet(PARQUET_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_save):,} rows  →  {PARQUET_PATH}")
print(f"File size : {PARQUET_PATH.stat().st_size / 1_048_576:.1f} MB")
print(f"Columns   : {df_save.columns.tolist()}")


Saved 4,631,261 rows  →  ../data/processed/bgl/20260921_2108/1_parser_bgl_annotated.parquet
File size : 242.3 MB
Columns   : ['label', 'is_anomaly', 'unix_ts', 'date', 'node_id', 'timestamp', 'component', 'subcomponent', 'level', 'raw', 'cluster_id', 'template', 'parameters', 'line_number']


In [10]:
parser.validate()


[INFO] Running template validation...
[INFO] Total lines parsed   : 4631261
[INFO] Distinct templates   : 180
[OK] 100% of lines received a template assignment.
[INFO] Template support (lines per template): min=1, max=1732239, avg=25729.2
[OK] Singleton template fraction looks reasonable.
[OK] No overly generic templates detected by wildcard heuristic.
[WARN] Found 7 templates that are single-use with no wildcards (likely overfitting).
      'NULL HARDWARE SEVERE NodeCard VPD chip is not accessible'
      'NULL DISCOVERY ERROR Node card status: ALERT 0, ALERT 1, ALERT 2, ALERT 3 is (are) active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is not asserted. TEMPERATURE MASK IS ACTIVE. No temperature error. Temperature Limit Error Latch is clear. PGOOD is asserted. PGOOD error latch is clear. MPGOOD is OK. MPGOOD error latch is clear. The 2.5 volt rail is OK. The 1.5 volt rail is OK.'
      'NULL SERV_NET WARNING DeclareServiceNetworkCharacter

## NEXT STEP: Enrich the mained templates with semantic information

Use LLM with sourced context to expand the template with semantic information for better embedding

See 2_EnrichTemplates.ipynb